# NEeds update to recalculate false negatives...figure out why infininty...

# Per-Muscle and Overall Average Metrics — All Algorithms

For each algorithm, reads its per-muscle result CSVs, computes the mean of every
metric across all scans, appends an `Overall_Mean` row, and saves a summary CSV.
The final cell combines all `Overall_Mean` rows into one comparison table.

In [10]:
import pathlib
import re
import pandas as pd
import numpy as np

In [11]:
import warnings

def process_algorithm(label, results_dir):
    """Return a summary DataFrame for one algorithm, or None if no CSVs found."""
    results_dir = pathlib.Path(results_dir)
    csv_files = sorted(results_dir.glob('*.csv'))

    if not csv_files:
        print(f'  [skip] no CSVs in {results_dir}')
        return None

    rows = []

    for csv_path in csv_files:
        muscle = extract_muscle(csv_path.stem)

        if muscle is None:
            print(f'  [skip] could not identify muscle in {csv_path.name}')
            continue

        df = pd.read_csv(csv_path, index_col=0)
        metric_vals = {}

        for col in df.columns:
            metric_name = canonical_metric(col)

            if metric_name is None:
                continue

            #s = pd.to_numeric(df[col], errors='coerce')
            #s = pd.to_numeric(df[col], errors='coerce').astype(np.float64).mean()
            s = pd.to_numeric(df[col], errors='coerce')

            # FORCE SAFE TYPE (this is critical)
            s = np.asarray(s, dtype=np.float64)
            
            # diagnostics BEFORE aggregation
            finite = np.isfinite(s)
            if finite.any():
                max_abs = np.nanmax(np.abs(s))
            
                if max_abs > 1e100:
                    print(
                        f"[VERY LARGE VALUES] "
                        f"{csv_path.name} | {col} | max_abs={max_abs:.3e}"
                    )
            
            if np.isinf(s).any():
                print(f"[INF FOUND] {csv_path.name} | {col}")
            
            # SAFE aggregation (no overflow possible in float64)
            metric_vals[metric_name] = np.nanmean(s, dtype=np.float64)
            # s = pd.to_numeric(df[col], errors='coerce').to_numpy(dtype=np.float64)
            # metric_vals[metric_name] = np.nanmean(s, dtype=np.float64)

            # # Diagnostic checks
            # try:
            #     finite_vals = s[np.isfinite(s)]

            #     if len(finite_vals):
            #         max_abs = np.abs(finite_vals).max()

            #         if max_abs > 1e100:
            #             print(
            #                 f'[VERY LARGE VALUES] '
            #                 f'file={csv_path.name} | '
            #                 f'column={col} | '
            #                 f'max_abs={max_abs:.3e}'
            #             )

            #     if np.isinf(s).any():
            #         print(
            #             f'[INF FOUND] '
            #             f'file={csv_path.name} | '
            #             f'column={col}'
            #         )

            # except Exception as e:
            #     print(
            #         f'[DIAGNOSTIC ERROR] '
            #         f'file={csv_path.name} | '
            #         f'column={col} | '
            #         f'error={e}'
            #     )

            # # Turn overflow warnings into exceptions so we know exactly where
            # try:
            #     with warnings.catch_warnings():
            #         warnings.filterwarnings(
            #             "error",
            #             message="overflow encountered in reduce"
            #         )

            #         metric_vals[metric_name] = s.mean()

            # except RuntimeWarning:
            #     print('\n' + '=' * 80)
            #     print('OVERFLOW DETECTED')
            #     print(f'File   : {csv_path}')
            #     print(f'Column : {col}')
            #     print(f'Metric : {metric_name}')
            #     print(f'Dtype  : {s.dtype}')
            #     print(f'Count  : {s.count()}')

            #     try:
            #         print(f'Min    : {s.min()}')
            #         print(f'Max    : {s.max()}')
            #     except Exception:
            #         pass

            #     print('\nFirst values:')
            #     print(s.head(54))

            #     print('=' * 80 + '\n')

            #     raise

        row = {'muscle': muscle}
        row.update(metric_vals)
        rows.append(row)

    if not rows:
        return None

    summary = pd.DataFrame(rows).set_index('muscle')
    summary.insert(0, 'algorithm', label)

    # Also protect the overall mean calculation
    try:
        with warnings.catch_warnings():
            warnings.filterwarnings(
                "error",
                message="overflow encountered in reduce"
            )

            overall = (
                summary
                .select_dtypes(include='number')
                .mean()
                .rename('Overall_Mean')
            )

    except RuntimeWarning:
        print('\nOVERFLOW OCCURRED DURING OVERALL SUMMARY MEAN')
        print(summary.select_dtypes(include='number').describe())
        raise

    overall['algorithm'] = label

    summary = pd.concat([summary, overall.to_frame().T])
    summary.index.name = 'muscle'

    return summary

In [12]:
import numpy as np
import pandas as pd
import warnings
import pathlib

_FLOAT64_MAX = np.finfo(np.float64).max

def process_algorithm(label, results_dir):
    results_dir = pathlib.Path(results_dir)
    csv_files = sorted(results_dir.glob("*.csv"))

    if not csv_files:
        print(f"[skip] no CSVs in {results_dir}")
        return None

    rows = []

    for csv_path in csv_files:
        muscle = extract_muscle(csv_path.stem)

        if muscle is None:
            print(f"[skip] could not identify muscle in {csv_path.name}")
            continue

        df = pd.read_csv(csv_path, index_col=0)
        metric_vals = {}

        for col in df.columns:
            metric_name = canonical_metric(col)
            if metric_name is None:
                continue

            # ── SAFE NUMERIC CONVERSION ─────────────────────────────
            s = pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=np.float64)

            # Replace inf/-inf AND float64-max sentinel values (stored overflow/inf
            # from source computation) with NaN so nanmean never overflows.
            s = np.where(np.isfinite(s) & (np.abs(s) < _FLOAT64_MAX), s, np.nan)

            # ── BASIC SANITY CHECKS ────────────────────────────────────
            finite = np.isfinite(s)
            if not finite.any():
                print(f"[ALL NON-FINITE] {csv_path.name} | {col}")
                continue

            vals = s[finite]
            min_val = np.min(vals)
            max_val = np.max(vals)

            if metric_name in {"dice", "jaccard", "volume_similarity"}:
                if max_val > 1.0 or min_val < -0.1:
                    print(
                        f"[SCALE ERROR] {csv_path.name} | {col} "
                        f"| expected ~[0,1], got min={min_val:.3f}, max={max_val:.3f}"
                    )

            if metric_name in {"false_negative", "false_positive"}:
                if max_val > 1e3:
                    print(
                        f"[COUNT-LIKE FN/FP] {csv_path.name} | {col} "
                        f"| max={max_val:.3e} (likely unnormalized)"
                    )

            # ── SAFE AGGREGATION ────────────────────────────────────
            metric_vals[metric_name] = np.nanmean(s, dtype=np.float64)

        # ── DERIVED: inter-slice dice ratio (pred / gt) ──────────────
        # Normalises prediction smoothness against the object's inherent
        # slice-to-slice variation; 1.0 = as consistent as the GT shape.
        pred = metric_vals.get("inter_slice_dice_pred")
        gt   = metric_vals.get("inter_slice_dice_gt")
        if pred is not None and gt is not None and gt > 0:
            metric_vals["inter_slice_dice_ratio"] = pred / gt

        row = {"muscle": muscle}
        row.update(metric_vals)
        rows.append(row)

    if not rows:
        return None

    summary = pd.DataFrame(rows).set_index("muscle")
    summary.insert(0, "algorithm", label)

    numeric = summary.select_dtypes(include="number").astype(np.float64)
    overall = numeric.mean().rename("Overall_Mean")
    overall["algorithm"] = label

    summary = pd.concat([summary, overall.to_frame().T])
    summary.index.name = "muscle"

    return summary

In [13]:
# for csv_path in csv_files:

#     df = pd.read_csv(csv_path, index_col=0)

#     metric_vals = {}

#     for col in df.columns:
#         ...

In [14]:
# for csv_path in csv_files:
#     df = pd.read_csv(csv_path, index_col=0)

#     metric_vals = {}
    
#     for col in df.columns:
#         metric_name = canonical_metric(col)
#         if metric_name is None:
#             continue
    
#         # ── SAFE NUMERIC CONVERSION ─────────────────────────────
#         s = pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=np.float64)
    
#         # ── REMOVE NON-FINITE VALUES FOR STABILITY ───────────────
#         finite = np.isfinite(s)
    
#         if not finite.any():
#             print(f"[ALL NON-FINITE] {csv_path.name} | {col}")
#             continue
        
#             vals = s[finite]
        
#             # ── BASIC DIAGNOSTICS ────────────────────────────────────
#             min_val = np.min(vals)
#             max_val = np.max(vals)
#             max_abs = np.max(np.abs(vals))
        
#             if np.isinf(s).any():
#                 print(f"[INF FOUND] {csv_path.name} | {col}")
        
#             if max_abs > 1e100:
#                 print(
#                     f"[VERY LARGE VALUES] {csv_path.name} | {col} | max_abs={max_abs:.3e}"
#                 )
        
#             # ── SCALE SANITY CHECKS ──────────────────────────────────
#             if metric_name in {"dice", "jaccard", "volume_similarity"}:
#                 if max_val > 1.0 or min_val < -0.1:
#                     print(
#                         f"[SCALE ERROR] {csv_path.name} | {col} "
#                         f"| min={min_val:.3f}, max={max_val:.3f}"
#                     )
        
#             if metric_name in {"false_negative", "false_positive"}:
#                 if max_val > 1e3:
#                     print(
#                         f"[LIKELY RAW COUNTS] {csv_path.name} | {col} "
#                         f"| max={max_val:.3e}"
#                     )
        
#             # ── SAFE AGGREGATION ─────────────────────────────────────
#             metric_vals[metric_name] = np.nanmean(s)

In [15]:
EVAL_DIR    = pathlib.Path(r'C:\Projects\dissector\eval_notebooks')
SUMMARY_DIR = EVAL_DIR / 'summary_results'
SUMMARY_DIR.mkdir(exist_ok=True)

# ── Canonical muscle names ────────────────────────────────────────────────────
MUSCLE_ALIASES = {
    'r_gracilis':  'R_gracilis',
    'l_gracilis':  'L_gracilis',
    'r_sartorius': 'R_sartorius',
    'l_sartorius': 'L_sartorius',
    'r_sart':      'R_sartorius',
    'l_sart':      'L_sartorius',
}

# ── Canonical metric names ────────────────────────────────────────────────────
# ORDER MATTERS — more specific patterns must come before general ones.
# inter_slice_dice_* must be before dice, otherwise the bare `dice` pattern
# would match them first (they contain the substring "dice").
_METRIC_RE = [
    ('inter_slice_dice_pred', re.compile(r'inter_slice_dice_pred',          re.I)),
    ('inter_slice_dice_gt',   re.compile(r'inter_slice_dice_gt',            re.I)),
    # Anchored: only matches when "dice" (or "lower_dice") is the entire bare column
    ('dice',                  re.compile(r'^(?:lower_)?dice$',              re.I)),
    ('hausdorff',             re.compile(r'hausdorff',                      re.I)),
    ('jaccard',               re.compile(r'jaccard',                        re.I)),
    ('volume_similarity',     re.compile(r'volume_similarity',              re.I)),
    ('false_negative',        re.compile(r'false.?neg|falseNeg',            re.I)),
    ('false_positive',        re.compile(r'false.?pos|falsePo',             re.I)),
    ('bce',                   re.compile(r'\bbce\b|binary_cross_entropy',   re.I)),
    ('boundary_iou_3d',       re.compile(r'boundary_iou',                  re.I)),
]

# Strip muscle-name prefix from a column so the pattern matches cleanly
_MUSCLE_PREFIX_RE = re.compile(r'^[RrLl]_(?:gracilis|sartorius|sart)_', re.I)

def canonical_metric(col):
    bare = _MUSCLE_PREFIX_RE.sub('', col).rstrip(':')
    for name, pat in _METRIC_RE:
        if pat.search(bare):
            return name
    return None


def extract_muscle(stem):
    s = stem.lower()
    for alias in sorted(MUSCLE_ALIASES, key=len, reverse=True):
        if f'_{alias}_' in s or s.endswith(f'_{alias}'):
            return MUSCLE_ALIASES[alias]
    return None


# def process_algorithm(label, results_dir):
#     """Return a summary DataFrame for one algorithm, or None if no CSVs found."""
#     results_dir = pathlib.Path(results_dir)
#     csv_files   = sorted(results_dir.glob('*.csv'))
#     if not csv_files:
#         print(f'  [skip] no CSVs in {results_dir}')
#         return None

#     rows = []
#     for csv_path in csv_files:
#         muscle = extract_muscle(csv_path.stem)
#         if muscle is None:
#             print(f'  [skip] could not identify muscle in {csv_path.name}')
#             continue

#         df = pd.read_csv(csv_path, index_col=0)
#         metric_vals = {}
#         for col in df.columns:
#             m = canonical_metric(col)
#             if m:
#                 metric_vals[m] = pd.to_numeric(df[col], errors='coerce').mean()
#                 if np.isfinite(m).any():
#                     max_abs = np.nanmax(np.abs(s))
#                     if max_abs > 1e100:
#                         print(
#                             f"Suspiciously large values: "
#                             f"{csv_path.name} | {col} | max_abs={max_abs:e}"
#                         )

                

#         row = {'muscle': muscle}
#         row.update(metric_vals)
#         rows.append(row)

#     if not rows:
#         return None

#     summary = pd.DataFrame(rows).set_index('muscle')
#     summary.insert(0, 'algorithm', label)

#     overall = summary.select_dtypes(include='number').mean().rename('Overall_Mean')
#     overall['algorithm'] = label
#     summary = pd.concat([summary, overall.to_frame().T])
#     summary.index.name = 'muscle'
#     return summary


# print('Helpers ready.')

In [20]:
# ── Algorithm registry ────────────────────────────────────────────────────────
REGISTRY = [
    ('MuscleMap Thigh (water)',           'muscle_map_thigh/results_water'),
    ('MuscleMap Thigh (fat fraction)',    'muscle_map_thigh/results_fat_frac'),
    ('MuscleMap WB (water)',              'muscle_map_wb/results_water'),
    ('MuscleMap WB (fat fraction)',       'muscle_map_wb/results_fat_frac'),
    ('MM WB + MedSAM bbox (water)',       'muscle_map_wb_boxes_medsam/results_water'),
    ('MM WB + MedSAM bbox (fat fraction)','muscle_map_wb_boxes_medsam/results_fatfrac'),
    ('MM WB + MedSAM mask (water)',        'muscle_map_wb_masks_medsam/results_water'),
    ('MM WB + MedSAM mask (fat fraction)', 'muscle_map_wb_masks_medsam/results_fat_frac'),
    ('MM WB + SLM-SAM2 (water)',          'muscle_map_wb+slmsam/results_water'),
    ('MM WB + SLM-SAM2 (fat fraction)',   'muscle_map_wb+slmsam/results_fat_frac'),
    ('Dafne (water)',                     'dafne/results_water'),
    ('Dafne (fat fraction)',              'dafne/results_fat_frac'),
    ('Dafne + MedSAM (water)',            'dafne_and_medsam/results_water'),
    ('Dafne + MedSAM (fat fraction)',     'dafne_and_medsam/results_fat_frac'),
    ('Hirriririir (water)',               'multimodal-multiethnic/results_water'),
    ('Hirriririir (fat fraction)',        'multimodal-multiethnic/results_fat_frac'),
    ('MuSeg (water)',                     'museg/results_water'),
    ('MuSeg (fat fraction)',              'museg/results_fat_frac'),
    ('MuSeg (dixon)',                     'museg/results_dixon'),
    ('MedCLIP-SAMv2 (water)',             'medclipsamv2/results_water'),
    ('MedCLIP-SAMv2 (fat fraction)',      'medclipsamv2/results_fat_frac'),
    ('MedCLIP-SAMv2 + MM Boxes (water)',        'medclipsamv2plusboxes/results_water'),
    ('MedCLIP-SAMv2 + MM Boxes (fat fraction)', 'medclipsamv2plusboxes/results_fat_frac'),
    ('MedCLIP-SAMv2 Text+Boxes (water)',        'medclipsamv2textboxes/results_water'),
    ('MedCLIP-SAMv2 Text+Boxes (fat fraction)', 'medclipsamv2textboxes/results_fat_frac'),
    ('MedSegDiff (both)',                'medsegdiff/results_channels'),
    #('MedSegDiff (water)',                'medsegdiff/results_water'),
    ('MedSegDiff (fat fraction)',         'medsegdiff/results_fat_frac'),
]

print(f'{len(REGISTRY)} entries in registry.')
for label, rdir in REGISTRY:
    path = EVAL_DIR / rdir
    n = len(list(path.glob('*.csv'))) if path.exists() else 0
    status = f'{n} CSVs' if path.exists() else 'DIR MISSING'
    print(f'  {label}: {status}')

27 entries in registry.
  MuscleMap Thigh (water): 4 CSVs
  MuscleMap Thigh (fat fraction): 4 CSVs
  MuscleMap WB (water): 4 CSVs
  MuscleMap WB (fat fraction): 4 CSVs
  MM WB + MedSAM bbox (water): 4 CSVs
  MM WB + MedSAM bbox (fat fraction): 4 CSVs
  MM WB + MedSAM mask (water): 4 CSVs
  MM WB + MedSAM mask (fat fraction): 4 CSVs
  MM WB + SLM-SAM2 (water): 4 CSVs
  MM WB + SLM-SAM2 (fat fraction): 4 CSVs
  Dafne (water): 4 CSVs
  Dafne (fat fraction): 4 CSVs
  Dafne + MedSAM (water): 4 CSVs
  Dafne + MedSAM (fat fraction): 4 CSVs
  Hirriririir (water): 4 CSVs
  Hirriririir (fat fraction): 4 CSVs
  MuSeg (water): 4 CSVs
  MuSeg (fat fraction): 4 CSVs
  MuSeg (dixon): 4 CSVs
  MedCLIP-SAMv2 (water): 4 CSVs
  MedCLIP-SAMv2 (fat fraction): 4 CSVs
  MedCLIP-SAMv2 + MM Boxes (water): DIR MISSING
  MedCLIP-SAMv2 + MM Boxes (fat fraction): DIR MISSING
  MedCLIP-SAMv2 Text+Boxes (water): 4 CSVs
  MedCLIP-SAMv2 Text+Boxes (fat fraction): 4 CSVs
  MedSegDiff (both): 5 CSVs
  MedSegDiff (fat fr

In [21]:
# ── Process all algorithms ────────────────────────────────────────────────────
summaries = {}  # label -> DataFrame

for label, rdir in REGISTRY:
    print(f'\n── {label} ──')
    df = process_algorithm(label, EVAL_DIR / rdir)
    if df is None:
        continue
    summaries[label] = df

    # Display
    num_cols = df.select_dtypes(include='number').columns
    display(df.reset_index().style.format('{:.4f}', subset=num_cols).hide(axis='index'))

    # Save individual summary CSV
    safe_name = re.sub(r'[^\w]+', '_', label).strip('_').lower()
    out_path  = SUMMARY_DIR / f'{safe_name}_avg_metrics.csv'
    df.to_csv(out_path, float_format='%.4f')
    print(f'  Saved -> {out_path}')

print(f'\nProcessed {len(summaries)}/{len(REGISTRY)} algorithms.')


── MuscleMap Thigh (water) ──
[SCALE ERROR] df_L_gracilis_musclemap_thigh_water.csv | L_gracilis_volume_similarity | expected ~[0,1], got min=-1.634, max=1.507
[SCALE ERROR] df_L_sartorius_musclemap_thigh_water.csv | L_sartorius_volume_similarity | expected ~[0,1], got min=-0.557, max=1.861
[SCALE ERROR] df_R_gracilis_musclemap_thigh_water.csv | R_gracilis_volume_similarity | expected ~[0,1], got min=-0.747, max=1.563
[SCALE ERROR] df_R_sartorius_musclemap_thigh_water.csv | R_sartorius_volume_similarity | expected ~[0,1], got min=-0.798, max=1.545


muscle,algorithm,dice,hausdorff,jaccard,volume_similarity,false_negative,false_positive,bce,boundary_iou_3d,inter_slice_dice_pred,inter_slice_dice_gt,inter_slice_dice_ratio
L_gracilis,MuscleMap Thigh (water),0.684325,28.766253,0.551945,0.208216,0.210326,0.000300,0.007047,0.467432,0.821180,0.833338,0.985410
L_sartorius,MuscleMap Thigh (water),0.604428,54.290201,0.472450,0.507593,0.159124,0.000553,0.010951,0.398355,0.792613,0.870424,0.910606
R_gracilis,MuscleMap Thigh (water),0.714389,17.886646,0.584419,0.122188,0.235825,0.000232,0.006464,0.488327,0.818031,0.833328,0.981644
R_sartorius,MuscleMap Thigh (water),0.703162,31.291812,0.557300,0.128245,0.234319,0.000317,0.008562,0.457097,0.828695,0.856942,0.967038
Overall_Mean,MuscleMap Thigh (water),0.676576,33.058728,0.541529,0.241560,0.209899,0.000350,0.008256,0.452803,0.815130,0.848508,0.961174


  Saved -> C:\Projects\dissector\eval_notebooks\summary_results\musclemap_thigh_water_avg_metrics.csv

── MuscleMap Thigh (fat fraction) ──
[SCALE ERROR] df_L_gracilis_musclemap_thigh_fatfrac.csv | L_gracilis_volume_similarity | expected ~[0,1], got min=-1.716, max=1.108
[SCALE ERROR] df_L_sartorius_musclemap_thigh_fatfrac.csv | L_sartorius_volume_similarity | expected ~[0,1], got min=-0.825, max=1.391
[SCALE ERROR] df_R_gracilis_musclemap_thigh_fatfrac.csv | R_gracilis_volume_similarity | expected ~[0,1], got min=-1.440, max=0.421
[SCALE ERROR] df_R_sartorius_musclemap_thigh_fatfrac.csv | R_sartorius_volume_similarity | expected ~[0,1], got min=-1.515, max=1.237


muscle,algorithm,dice,hausdorff,jaccard,volume_similarity,false_negative,false_positive,bce,boundary_iou_3d,inter_slice_dice_pred,inter_slice_dice_gt,inter_slice_dice_ratio
L_gracilis,MuscleMap Thigh (fat fraction),0.711086,16.144738,0.567603,-0.048961,0.276950,0.000213,0.006964,0.462193,0.851521,0.830722,1.025037
L_sartorius,MuscleMap Thigh (fat fraction),0.701606,35.103649,0.557453,0.089333,0.227949,0.000338,0.009527,0.448979,0.860728,0.867966,0.991660
R_gracilis,MuscleMap Thigh (fat fraction),0.772608,13.065069,0.648985,-0.200498,0.272155,0.000106,0.005446,0.540799,0.855582,0.823542,1.038906
R_sartorius,MuscleMap Thigh (fat fraction),0.728916,28.507110,0.592696,-0.090720,0.273982,0.000201,0.008186,0.482131,0.865860,0.841052,1.029496
Overall_Mean,MuscleMap Thigh (fat fraction),0.728554,23.205141,0.591684,-0.062711,0.262759,0.000215,0.007531,0.483525,0.858423,0.840821,1.021275


  Saved -> C:\Projects\dissector\eval_notebooks\summary_results\musclemap_thigh_fat_fraction_avg_metrics.csv

── MuscleMap WB (water) ──
[SCALE ERROR] df_L_gracilis_musclemap_wb_water.csv | L_gracilis_volume_similarity | expected ~[0,1], got min=-1.846, max=0.199
[SCALE ERROR] df_L_sartorius_musclemap_wb_water.csv | L_sartorius_volume_similarity | expected ~[0,1], got min=-0.672, max=0.260
[SCALE ERROR] df_R_gracilis_musclemap_wb_water.csv | R_gracilis_volume_similarity | expected ~[0,1], got min=-1.783, max=0.117
[SCALE ERROR] df_R_sartorius_musclemap_wb_water.csv | R_sartorius_volume_similarity | expected ~[0,1], got min=-0.810, max=0.099


...

  Saved -> C:\Projects\dissector\eval_notebooks\summary_results\medclip_samv2_text_boxes_water_avg_metrics.csv

── MedCLIP-SAMv2 Text+Boxes (fat fraction) ──
[SCALE ERROR] df_L_gracilis_mcsam2textboxes_fatfrac.csv | L_gracilis_volume_similarity | expected ~[0,1], got min=-2.000, max=-0.499
[SCALE ERROR] df_L_sartorius_mcsam2textboxes_fatfrac.csv | L_sartorius_volume_similarity | expected ~[0,1], got min=-1.980, max=-0.056
[SCALE ERROR] df_R_gracilis_mcsam2textboxes_fatfrac.csv | R_gracilis_volume_similarity | expected ~[0,1], got min=-2.000, max=-0.506
[SCALE ERROR] df_R_sartorius_mcsam2textboxes_fatfrac.csv | R_sartorius_volume_similarity | expected ~[0,1], got min=-1.975, max=-0.363


muscle,algorithm,dice,hausdorff,jaccard,volume_similarity,false_negative,false_positive,bce,boundary_iou_3d,inter_slice_dice_pred,inter_slice_dice_gt,inter_slice_dice_ratio
L_gracilis,MedCLIP-SAMv2 Text+Boxes (fat fraction),0.088391,335.664928,0.060857,-1.816591,0.938593,0.000069,1.175002,0.030749,0.792162,0.830722,0.953583
L_sartorius,MedCLIP-SAMv2 Text+Boxes (fat fraction),0.314782,242.042752,0.219323,-1.325922,0.772184,0.000075,0.196745,0.132188,0.826223,0.867966,0.951907
R_gracilis,MedCLIP-SAMv2 Text+Boxes (fat fraction),0.086150,348.778296,0.057940,-1.821781,0.941563,0.000055,1.190292,0.029552,0.784463,0.823542,0.952548
R_sartorius,MedCLIP-SAMv2 Text+Boxes (fat fraction),0.332937,246.578827,0.233057,-1.286039,0.758573,0.000057,0.168054,0.148657,0.822379,0.841052,0.977798
Overall_Mean,MedCLIP-SAMv2 Text+Boxes (fat fraction),0.205565,293.266201,0.142794,-1.562584,0.852728,0.000064,0.682523,0.085287,0.806307,0.840821,0.958959


  Saved -> C:\Projects\dissector\eval_notebooks\summary_results\medclip_samv2_text_boxes_fat_fraction_avg_metrics.csv

── MedSegDiff (both) ──
[SCALE ERROR] df_R_gracilis_medsegdiff_both.csv | R_gracilis_volume_similarity | expected ~[0,1], got min=-2.000, max=-1.959


muscle,algorithm,dice,hausdorff,jaccard,volume_similarity,false_negative,false_positive,bce,boundary_iou_3d,inter_slice_dice_pred,inter_slice_dice_gt,inter_slice_dice_ratio
L_gracilis,MedSegDiff (both),nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
L_sartorius,MedSegDiff (both),nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
R_gracilis,MedSegDiff (both),0.003600,464.251718,0.001805,-1.984693,0.998191,0.000508,3.358313,0.001132,0.134308,0.833328,0.161171
R_gracilis,MedSegDiff (both),nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
R_sartorius,MedSegDiff (both),nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
Overall_Mean,MedSegDiff (both),0.003600,464.251718,0.001805,-1.984693,0.998191,0.000508,3.358313,0.001132,0.134308,0.833328,0.161171


  Saved -> C:\Projects\dissector\eval_notebooks\summary_results\medsegdiff_both_avg_metrics.csv

── MedSegDiff (fat fraction) ──
[skip] no CSVs in C:\Projects\dissector\eval_notebooks\medsegdiff\results_fat_frac

Processed 24/27 algorithms.


In [22]:
# ── Combined Overall Means ────────────────────────────────────────────────────
overall_rows = [
    df.loc[['Overall_Mean']]
    for df in summaries.values()
    if 'Overall_Mean' in df.index
]

combined = pd.concat(overall_rows)
combined.index = [row['algorithm'] for _, row in combined.iterrows()]
combined.index.name = 'algorithm'
combined = combined.drop(columns='algorithm')
combined = combined.drop(columns=['inter_slice_dice_pred', 'inter_slice_dice_gt'], errors='ignore')

num_cols = combined.select_dtypes(include='number').columns
display(
    combined.reset_index()
    .style
    .format('{:.4f}', subset=num_cols)
    .hide(axis='index')
    .background_gradient(subset=['dice'], cmap='RdYlGn', axis=0)
    .background_gradient(subset=['hausdorff'], cmap='RdYlGn_r', axis=0)
)

out_combined = SUMMARY_DIR / 'overall_means.csv'
combined.to_csv(out_combined, float_format='%.4f')
print('Saved ->', out_combined)

algorithm,dice,hausdorff,jaccard,volume_similarity,false_negative,false_positive,bce,boundary_iou_3d,inter_slice_dice_ratio
MuscleMap Thigh (water),0.676576,33.058728,0.541529,0.241560,0.209899,0.000350,0.008256,0.452803,0.961174
MuscleMap Thigh (fat fraction),0.728554,23.205141,0.591684,-0.062711,0.262759,0.000215,0.007531,0.483525,1.021275
MuscleMap WB (water),0.824880,16.829963,0.710138,-0.125285,0.208108,0.000104,0.005153,0.584924,1.047251
MuscleMap WB (fat fraction),0.800018,16.855175,0.679919,-0.189001,0.244172,0.000096,0.005720,0.562860,1.054587
MM WB + MedSAM bbox (water),0.736643,17.677685,0.592000,-0.427191,0.381743,0.000054,0.009666,0.411404,1.044403
MM WB + MedSAM bbox (fat fraction),0.661686,18.489591,0.504340,-0.369663,0.419796,0.000134,0.011162,0.403860,0.903552
MM WB + MedSAM mask (water),0.368572,100.448587,0.248897,-0.785693,0.674892,0.000401,0.055357,0.175215,1.002272
MM WB + MedSAM mask (fat fraction),0.333143,105.612221,0.216759,-0.791504,0.714850,0.000432,0.048787,0.145804,0.918781
MM WB + SLM-SAM2 (water),0.379870,103.931503,0.249396,0.147030,0.524766,0.000553,0.019909,0.262258,0.760432
MM WB + SLM-SAM2 (fat fraction),0.383647,100.641315,0.260335,-0.117534,0.577813,0.000415,0.024271,0.201519,0.780216


Saved -> C:\Projects\dissector\eval_notebooks\summary_results\overall_means.csv


In [ ]:
# we have an overflow problem....we can manually replace inf with 1 if it can't be fixed? 